In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [2]:
dat = gpd.read_file('neigh_reviews_subset.gpkg')

In [3]:
dat['text'] = dat.text.str.lower()
dat = dat.dropna(subset=['text', 'gentrified']).reset_index(drop=True)

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import re

# 1. Setup (assuming 'undf' is a DataFrame)
# corpus_undf <- corpus(undf, text_field = "text")
texts = dat['text']

# 2. Preprocessing Function (tokens, remove_punct, remove_numbers, wordstem)
stemmer = SnowballStemmer("english")
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # remove_punct and remove_numbers using regex
    #text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # tokens_wordstem and stopwords remove
    tokens = [stemmer.stem(word) for word in text.split() if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing
processed_texts = texts.apply(preprocess)

# 3. Create DFM (Document-Feature Matrix)
# dfm <- dfm(toks)
vectorizer = CountVectorizer()
dfm = vectorizer.fit_transform(processed_texts)

# 4. Trim DFM (min_docfreq = 0.05)
# dfm_trim(dfm, min_docfreq = 0.05, docfreq_type = "prop")
trimmed_vectorizer = CountVectorizer(min_df=0.05)
dfm_trimmed = trimmed_vectorizer.fit_transform(processed_texts)

# Convert to DataFrame for viewing (optional)
dfm_trimmed_df = pd.DataFrame(
    dfm_trimmed.toarray(), 
    columns=trimmed_vectorizer.get_feature_names_out()
)

print(dfm_trimmed_df)

        actual  also  alway  amaz  anoth  appet  area  around  ask  atmospher  \
0            0     0      0     0      0      0     1       0    0          0   
1            0     0      1     0      0      0     0       0    0          0   
2            0     1      1     0      0      0     0       0    0          0   
3            0     1      0     0      0      0     0       0    0          0   
4            0     0      0     0      0      0     0       0    0          0   
...        ...   ...    ...   ...    ...    ...   ...     ...  ...        ...   
681784       0     1      0     0      0      0     0       0    0          0   
681785       0     1      0     2      0      0     0       0    0          0   
681786       0     1      0     1      0      0     0       0    0          0   
681787       0     1      0     0      0      0     0       0    0          0   
681788       0     1      0     0      2      0     0       0    0          0   

        ...  wasnt  way  we

In [5]:
dat['id'] = dat.index

In [49]:
from sklearn.model_selection import train_test_split
training, validation = train_test_split(
    dat, 
    train_size=0.75, 
    random_state=32123
)


In [ ]:
#Create separate subsets for DFM
dfmat_train = dfm_trimmed_df.loc[training.index]
dfmat_val = dfm_trimmed_df.loc[validation.index]
X_train = dfmat_train
X_test = dfmat_val
y_train = dat.loc[training.index].gentrified
y_test = dat.loc[validation.index].gentrified

In [9]:
dfmat_train

,actual,also,alway,amaz,anoth,appet,area,around,ask,atmospher,...,wasnt,way,week,well,went,work,worth,would,year,your
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
511336,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
511337,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
511338,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
511339,0,0,0,1,0,0,0,0,0,1,...,1,0,0,1,0,0,0,2,0,0


In [40]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import confusion_matrix, classification_report

lasso_model = LogisticRegression(penalty='l1', solver = 'saga', random_state=4323, )
lasso_model.fit(X_train, y_train)

testing = []
for i, j in enumerate(X_train.columns):
    testing.append({
        'word': j,
        'rank': lasso_model.coef_[0][i],
        'rank_abs': np.abs(lasso_model.coef_[0][i])
    })
importance = pd.DataFrame(testing)
importance = importance.sort_values(by = 'rank', ascending = False)

# --- 4. Out of Sample Predictions ---
# Probability predictions (type="response")
pred_probs = lasso_model.predict_proba(X_test)[:, 1]

# Confusion Matrix (Table)
# Since R's index 40 or 1 refers to specific lambdas, in Python you'd just 
# predict using the fitted model or specific thresholds.
print(confusion_matrix(y_test, lasso_model.predict(X_test)))

confusion_matrix(y_test, lasso_model.predict(X_test))

[[155653     42]
 [ 14741     12]]


array([[155653,     42],
       [ 14741,     12]])

In [48]:
print(classification_report(y_test, lasso_model.predict(X_test)))

              precision    recall  f1-score   support

         0.0       0.91      1.00      0.95    155695
         1.0       0.22      0.00      0.00     14753

    accuracy                           0.91    170448
   macro avg       0.57      0.50      0.48    170448
weighted avg       0.85      0.91      0.87    170448



In [52]:
# --- 5. Cross-Validation (cv.glmnet) ---
# Cs is the number of inverse-lambdas to try
cv_model = LogisticRegressionCV(
    penalty='l1', 
    solver='saga', 
    cv=5, 
    scoring='accuracy', 
    class_weight='balanced' # Replaces manual weights
)
cv_model.fit(X_train, y_train)

testing2 = []
for i, j in enumerate(X_train.columns):
    testing2.append({
        'word': j,
        'rank': cv_model.coef_[0][i],
        'rank_abs': np.abs(cv_model.coef_[0][i])
    })
importance2 = pd.DataFrame(testing2)
importance2 = importance2.sort_values(by = 'rank', ascending = False)

pred_val = cv_model.predict(X_test)
tab_val = confusion_matrix(y_test, pred_val)
print("Final Validation Table:\n", tab_val)
importance2.head()

Final Validation Table:
 [[144054  11641]
 [ 12974   1779]]


,word,rank,rank_abs
16,beer,0.076506,0.076506
0,actual,0.000000,0.000000
135,right,0.000000,0.000000
125,pretti,0.000000,0.000000
126,price,0.000000,0.000000


In [53]:
print(classification_report(y_test, cv_model.predict(X_test)))

              precision    recall  f1-score   support

         0.0       0.92      0.93      0.92    155695
         1.0       0.13      0.12      0.13     14753

    accuracy                           0.86    170448
   macro avg       0.52      0.52      0.52    170448
weighted avg       0.85      0.86      0.85    170448



In [54]:
importance2.tail()

,word,rank,rank_abs
69,give,0.000000,0.000000
70,go,0.000000,0.000000
71,good,0.000000,0.000000
196,your,0.000000,0.000000
133,restaur,-0.006536,0.006536
